## MLP training (Breast Cancer Wisconsin)
This notebook trains a simple `MLPClassifier` on the prepared splits in `data_processed/` and reports **accuracy** on `train/val/test`.
**Outputs** are saved to `artifacts/mlp/`:
- `model.joblib`
- `metrics.json`

In [25]:
from pathlib import Path
import json

import joblib
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier


In [26]:
DATA_DIR = Path("data_processed")
OUT_DIR = Path("artifacts/mlp")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

(X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape)


((364, 30), (364,), (91, 30), (91,), (114, 30), (114,))

In [27]:
SEED = 42

# A bit more strict / regularized version:
# - early stopping on an internal validation split
# - stronger L2 regularization (alpha)
# - keep the same architecture for comparability
model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    tol=1e-4,
    random_state=SEED,
)

model.fit(X_train, y_train)

def acc(X, y):
    return float(accuracy_score(y, model.predict(X)))

metrics = {
    "accuracy": {
        "train": acc(X_train, y_train),
        "val": acc(X_val, y_val),
        "test": acc(X_test, y_test),
    },
    "n_iter_": int(getattr(model, "n_iter_", -1)),
    "loss_": float(getattr(model, "loss_", np.nan)),
    "classes_": [int(c) for c in getattr(model, "classes_", [])],
    "seed": SEED,
    "mlp": {
        "hidden_layer_sizes": [100, 50],
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "max_iter": 2000,
        "early_stopping": True,
        "validation_fraction": 0.1,
        "n_iter_no_change": 20,
        "tol": 1e-4,
    },
    "data_dir": str(DATA_DIR),
}

metrics


{'accuracy': {'train': 0.9917582417582418,
  'val': 0.978021978021978,
  'test': 0.9473684210526315},
 'n_iter_': 57,
 'loss_': 0.03282367196985429,
 'classes_': [0, 1],
 'seed': 42,
 'mlp': {'hidden_layer_sizes': [100, 50],
  'activation': 'relu',
  'solver': 'adam',
  'alpha': 0.001,
  'learning_rate_init': 0.001,
  'max_iter': 2000,
  'early_stopping': True,
  'validation_fraction': 0.1,
  'n_iter_no_change': 20,
  'tol': 0.0001},
 'data_dir': 'data_processed'}

In [28]:
joblib.dump(model, OUT_DIR / "model.joblib")
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", (OUT_DIR / "model.joblib").as_posix())
print("Saved:", (OUT_DIR / "metrics.json").as_posix())


Saved: artifacts/mlp/model.joblib
Saved: artifacts/mlp/metrics.json
